# Problem 2: Exploratory Data Analysis of Washing Machine Use

**Objective:**
In this notebook, we perform exploratory data analysis (EDA) of washing machine electrical consumption across **4 REFIT households** at 1-minute resolution:
- **House 1:** Beko WMC6140 (`Appliance 5`)
- **House 2:** LG F1289TD (`Appliance 2`)
- **House 3:** Washing Machine (`Appliance 6`)
- **House 5:** AEG L99695HWD (`Appliance 3`)

### Key Deliverables Addressed:
1. **Plain Language Cycle Detection Rule:** Define a physical, intuitive rule to detect discrete washing machine cycles.
2. **Data Availability & Quality Audit:** Quantify monitored duration, missing gaps, off/zero time, and cycle counts.
3. **Usage Patterns:** Analyze usage by hour of day (diurnal) and day of week.
4. **Cross-Household Signatures:** Compare cycle duration, peak power, and energy per cycle (kWh).
5. **Visualizations:**
   - (i) Representative 24-hour aggregate + washing machine trace
   - (ii) Zoomed-in view of detected cycles with start/end markers
   - (iii) Cross-house comparison boxplots and usage distributions

In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure figures directory exists
os.makedirs('figures', exist_ok=True)
print('Libraries imported successfully!')

## 1. Household Selection & Configuration
We configure the 4 homes, selecting their corresponding washing machine IAM channels from the official REFIT documentation.

In [ ]:
PARENT_DIR = '..'

HOUSE_CONFIG = {
    'House 1': {
        'file': os.path.join(PARENT_DIR, 'CLEAN_House1.csv'),
        'wm_col': 'Appliance5',
        'appliance_name': 'Beko WMC6140'
    },
    'House 2': {
        'file': os.path.join(PARENT_DIR, 'CLEAN_House2.csv'),
        'wm_col': 'Appliance2',
        'appliance_name': 'LG F1289TD'
    },
    'House 3': {
        'file': os.path.join(PARENT_DIR, 'CLEAN_House3.csv'),
        'wm_col': 'Appliance6',
        'appliance_name': 'Washing Machine'
    },
    'House 5': {
        'file': os.path.join(PARENT_DIR, 'CLEAN_House5.csv'),
        'wm_col': 'Appliance3',
        'appliance_name': 'AEG L99695HWD'
    }
}

## 2. Active Cycle Detection Rule (In Plain Language)
A domestic washing machine operates through distinct phases: drum fill, motor rotation, water heating, rinsing, and high-speed centrifugal spin.

**Our Rule:**
1. **Active Power Threshold ($P \ge 20$ W):** Distinguishes active motor agitation and electronics from idle standby ($< 5$ W).
2. **Pause / Gap Tolerance ($\le 5$ minutes):** Internal pauses between rinse and spin (e.g. while draining water) do not break the cycle.
3. **Minimum Duration ($\ge 15$ minutes):** Rejects short accidental button clicks or transient door openings.
4. **Minimum Peak Power ($\ge 500$ W):** Confirms that either the electric heating element or high-speed spin motor engaged.

In [ ]:
def detect_washing_cycles(power_1min, p_thresh=20.0, max_gap=5, min_dur=15, min_peak=500.0):
    times = power_1min.index
    values = power_1min.values
    cycles = []
    
    in_cycle = False
    start_idx = None
    gap_count = 0
    
    for i in range(len(values)):
        val = values[i]
        if np.isnan(val):
            val = 0.0
            
        if val >= p_thresh:
            if not in_cycle:
                in_cycle = True
                start_idx = i
            gap_count = 0
        else:
            if in_cycle:
                gap_count += 1
                if gap_count > max_gap or i == len(values) - 1:
                    end_idx = i - gap_count
                    dur = end_idx - start_idx + 1
                    cycle_vals = values[start_idx:end_idx+1]
                    peak = np.nanmax(cycle_vals) if len(cycle_vals) > 0 else 0.0
                    
                    if dur >= min_dur and peak >= min_peak:
                        energy_kwh = (np.nansum(cycle_vals) / 60.0) / 1000.0
                        cycles.append({
                            'start_time': times[start_idx],
                            'end_time': times[end_idx],
                            'duration_min': dur,
                            'peak_watts': peak,
                            'mean_watts': float(np.nanmean(cycle_vals)),
                            'energy_kwh': energy_kwh,
                            'start_hour': times[start_idx].hour,
                            'day_of_week': times[start_idx].day_name()
                        })
                    in_cycle = False
                    gap_count = 0
                    
    return pd.DataFrame(cycles)

## 3. Data Loading, Resampling, and Quality Metrics Extraction

In [ ]:
house_summaries = []
all_detected_cycles = []
house_resampled_dict = {}

for house_name, cfg in HOUSE_CONFIG.items():
    print(f'Processing {house_name} ({cfg["appliance_name"]})...')
    df = pd.read_csv(
        cfg['file'],
        usecols=['Time', 'Aggregate', cfg['wm_col']],
        dtype={'Aggregate': 'float32', cfg['wm_col']: 'float32'}
    )
    df['DateTime'] = pd.to_datetime(df['Time'])
    df.set_index('DateTime', inplace=True)
    df.drop(columns=['Time'], inplace=True)
    
    # Resample to 1-minute
    df_1min = df.resample('1min').mean()
    df_1min['Aggregate_clean'] = df_1min['Aggregate'].ffill(limit=5)
    df_1min['wm_clean'] = df_1min[cfg['wm_col']].ffill(limit=5)
    
    house_resampled_dict[house_name] = df_1min[['Aggregate_clean', 'wm_clean']].copy()
    
    # Quality Metrics
    total_minutes = len(df_1min)
    missing_minutes = df_1min['wm_clean'].isna().sum()
    missing_pct = (missing_minutes / total_minutes) * 100.0
    
    valid_wm = df_1min['wm_clean'].dropna()
    off_pct = ((valid_wm < 20.0).sum() / len(valid_wm)) * 100.0
    
    # Cycles
    cycles_df = detect_washing_cycles(df_1min['wm_clean'])
    cycles_df['house'] = house_name
    all_detected_cycles.append(cycles_df)
    
    duration_days = (df_1min.index.max() - df_1min.index.min()).total_seconds() / 86400.0
    cycles_per_week = (len(cycles_df) / duration_days) * 7.0
    
    house_summaries.append({
        'House': house_name,
        'Appliance Model': cfg['appliance_name'],
        'Start Date': df_1min.index.min().strftime('%Y-%m-%d'),
        'End Date': df_1min.index.max().strftime('%Y-%m-%d'),
        'Days Monitored': round(duration_days, 1),
        'Missing Data (%)': round(missing_pct, 2),
        'Off/Zero Time (%)': round(off_pct, 2),
        'Total Cycles': len(cycles_df),
        'Cycles / Week': round(cycles_per_week, 2),
        'Median Duration (min)': round(cycles_df['duration_min'].median(), 1),
        'Median Peak Power (W)': round(cycles_df['peak_watts'].median(), 1),
        'Median Energy (kWh)': round(cycles_df['energy_kwh'].median(), 3)
    })
    del df, df_1min
    gc.collect()

summary_table = pd.DataFrame(house_summaries)
all_cycles_df = pd.concat(all_detected_cycles, ignore_index=True)
display(summary_table)

## 4. Plot (i): Representative 24-Hour Aggregate + Washing Machine Trace
We examine a 24-hour trace for House 1 on October 23, 2013.

In [ ]:
trace_data = house_resampled_dict['House 1'].loc['2013-10-23 00:00:00':'2013-10-23 23:59:00']

plt.figure(figsize=(14, 5))
plt.plot(trace_data.index, trace_data['Aggregate_clean'], label='Total Household Aggregate Power', color='#1f77b4', lw=1.2, alpha=0.85)
plt.plot(trace_data.index, trace_data['wm_clean'], label='Washing Machine Power (Beko WMC6140)', color='#d62728', lw=1.8)
plt.fill_between(trace_data.index, 0, trace_data['wm_clean'], color='#d62728', alpha=0.2)
plt.title('Plot (i): Representative 24-Hour Power Trace (House 1 - October 23, 2013)', fontsize=12, fontweight='bold')
plt.ylabel('Power (Watts)', fontsize=11)
plt.xlabel('Time of Day', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.savefig('figures/problem2_24h_trace.png', dpi=200)
plt.show()

## 5. Plot (ii): Zoomed-In Detected Washing Machine Cycles
Demonstrating cycle boundary detection on the morning wash runs.

In [ ]:
cycle_zoom = trace_data.loc['2013-10-23 05:45:00':'2013-10-23 07:30:00']
plt.figure(figsize=(12, 5))
plt.plot(cycle_zoom.index, cycle_zoom['wm_clean'], color='#d62728', lw=2, marker='o', markersize=3, label='Washing Machine Load')
plt.axhline(20, color='gray', linestyle=':', label='Active Threshold (20 W)')
plt.axhline(500, color='purple', linestyle='--', alpha=0.7, label='Peak Qualification Threshold (500 W)')

h1_cycles = all_cycles_df[(all_cycles_df['house'] == 'House 1') & 
                          (all_cycles_df['start_time'] >= '2013-10-23 05:45:00') & 
                          (all_cycles_df['start_time'] <= '2013-10-23 07:30:00')]

for idx, c in h1_cycles.iterrows():
    plt.axvspan(c['start_time'], c['end_time'], color='green', alpha=0.15, 
                label='Detected Cycle' if idx == h1_cycles.index[0] else '')
    plt.text(c['start_time'], c['peak_watts'] + 50, f"{c['duration_min']}m | {c['peak_watts']:.0f}W", 
             fontsize=9, fontweight='bold', color='darkgreen')

plt.title('Plot (ii): Detected Washing Machine Operating Cycles (House 1 Zoom)', fontsize=12, fontweight='bold')
plt.ylabel('Power (Watts)', fontsize=11)
plt.xlabel('Timestamp', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.savefig('figures/problem2_detected_cycles.png', dpi=200)
plt.show()

## 6. Plot (iii): Cross-House Comparison
We compare:
1. **Cycle Duration** across all 4 houses (boxplot).
2. **Energy Consumed per Cycle** (boxplot).
3. **Hour of Day Usage Distribution**.
4. **Day of Week Usage Distribution**.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
houses_list = ['House 1', 'House 2', 'House 3', 'House 5']

# Duration
dur_data = [all_cycles_df[all_cycles_df['house'] == h]['duration_min'] for h in houses_list]
axes[0, 0].boxplot(dur_data, tick_labels=houses_list, showmeans=True, patch_artist=True,
                   boxprops=dict(facecolor='#c6dbef', color='#08519c'),
                   medianprops=dict(color='red', lw=2))
axes[0, 0].set_title('A. Cycle Duration Across Households', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Duration (Minutes)')
axes[0, 0].grid(True, linestyle='--', alpha=0.5)

# Energy
energy_data = [all_cycles_df[all_cycles_df['house'] == h]['energy_kwh'] for h in houses_list]
axes[0, 1].boxplot(energy_data, tick_labels=houses_list, showmeans=True, patch_artist=True,
                   boxprops=dict(facecolor='#c7e9c0', color='#006d2c'),
                   medianprops=dict(color='red', lw=2))
axes[0, 1].set_title('B. Energy per Cycle Across Households', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Energy (kWh)')
axes[0, 1].grid(True, linestyle='--', alpha=0.5)

# Hour of Day
for h in houses_list:
    h_cycles = all_cycles_df[all_cycles_df['house'] == h]
    hour_counts = h_cycles['start_hour'].value_counts().reindex(range(24), fill_value=0)
    hour_pct = (hour_counts / len(h_cycles)) * 100.0
    axes[1, 0].plot(range(24), hour_pct, marker='o', label=h, lw=1.8)

axes[1, 0].set_title('C. Usage by Hour of Day (% of Total Cycles)', fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel('Hour of Day (0-23)')
axes[1, 0].set_ylabel('Frequency (%)')
axes[1, 0].set_xticks(range(0, 24, 2))
axes[1, 0].grid(True, linestyle='--', alpha=0.5)
axes[1, 0].legend()

# Day of Week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
for h in houses_list:
    h_cycles = all_cycles_df[all_cycles_df['house'] == h]
    day_counts = h_cycles['day_of_week'].value_counts().reindex(day_order, fill_value=0)
    day_pct = (day_counts / len(h_cycles)) * 100.0
    axes[1, 1].plot(range(7), day_pct, marker='s', label=h, lw=1.8)

axes[1, 1].set_title('D. Usage by Day of Week (% of Total Cycles)', fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel('Day of Week')
axes[1, 1].set_ylabel('Frequency (%)')
axes[1, 1].set_xticks(range(7))
axes[1, 1].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
axes[1, 1].grid(True, linestyle='--', alpha=0.5)
axes[1, 1].legend()

plt.suptitle('Plot (iii): Cross-House Washing Machine Usage & Signature Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/problem2_cross_house_comparison.png', dpi=200)
plt.show()